# sPHENIX 3D IBF: prepare input for C++ Condor field calculation

This notebook builds the asymmetric 3D ion-backflow charge distribution and Green-function mode tables, but does **not** calculate the electric field. It writes one self-contained ROOT file that independent C++ Condor jobs can read.


In [ ]:
import re, json, time
from pathlib import Path
import numpy as np
import ROOT
from scipy import optimize, special, integrate
import matplotlib.pyplot as plt

ROOT.gROOT.SetBatch(True)
EPSILON_0 = 8.8541878128e-12
CM_TO_M = 1.0e-2
M_TO_CM = 100.0


## Configuration

In [ ]:
# Input files
PAD_PLACEMENT_FILE = Path("TPC_pad_placement.txt")
GAIN_MAP_FILE = Path("dbb9e854-b14e-488e-9991-8f6f1962dffb.txt")

GAIN_METHOD = 1                         # 1 = merged layer histograms; 2 = summed stored fits
MULTIPLY_CHARGE_BY_GAIN = True
NORMALIZE_GAIN_WEIGHTED_TOTAL = True

# Radial model
GENERATE_RADIAL_STUDY = True
FIT_GENERATED_CHARGE = True             # False -> use FIXED_ALPHA/FIXED_CONSTANT
GENERATION_MODE = "realistic_eta"       # "flat_eta", "realistic_eta", "isotropic_sphere"
ETA_DISTRIBUTION = "double_hump"

N_GENERATED = 2_000_000
ETA_MAX = 6.0
Z_GEM_CM = 102.325
FIT_R_MIN_CM = 22.8132065669
FIT_R_MAX_CM = 75.9666751734

# Approximate 200 GeV p+p dN/deta shape.
ETA_PEAK = 2.15
ETA_PEAK_WIDTH = 1.35
ETA_HUMP_AMPLITUDE = 0.35
ETA_CENTRAL_DIP = 0.08
ETA_CENTRAL_WIDTH = 0.55
ETA_EDGE_START = 4.0
ETA_EDGE_WIDTH = 0.65

FIXED_ALPHA = np.array([1.2, 1.2, 1.2], dtype=float)
FIXED_CONSTANT = np.array([1.0, 1.0, 1.0], dtype=float)

# TPC electrostatic volume and GEM/module boundaries.
A_CM, B_CM, L_CM = 20.0, 75.911, 102.325

# R1 includes the antenna-pad GEM area as requested.
MODULE_R_BOUNDS_CM = np.array([
    [22.8132065669, 40.0513196955],
    [41.6592025362 - 0.5*1.3179431693, 56.9695394315 + 0.5*1.3179431693],
    [58.9109633493 - 0.5*1.3175606263, 75.3666751734 + 0.5*1.3175606263],
])

# Production grid and mode configuration.
# Keep these identical to the C++ run configuration.
NR_SRC, NPHI_SRC, NZ_SRC = 36, 48, 40
NR_OBS, NPHI_OBS, NZ_OBS = 36, 48, 40

M_PHI_MAX = 12
N_RADIAL_MODES = 14
N_LONGITUDINAL_MODES = 14

RHO_REFERENCE_NC_PER_M3 = 20.0
K_EFF = 1.0

# Condor input output.
CONDOR_INPUT_ROOT = "sphenix_3d_ibf_condor_input.root"
SAVE_CHARGE_TH3 = True

# Precompute source-side basis values once here instead of repeating
# Bessel/trigonometric evaluations in every Condor job.
PRECOMPUTE_SOURCE_BASIS = True
SOURCE_BASIS_DTYPE = np.float32


## Parse real sector/module geometry and module gain map

In [ ]:
def wrap_phi(phi):
    return (phi + np.pi) % (2*np.pi) - np.pi

def parse_pad_geometry(path):
    text = Path(path).read_text(errors="ignore")
    region = None
    data = {}
    for line in text.splitlines():
        m = re.search(r"region\s+(\d+)\s+first_layer", line)
        if m:
            region = int(m.group(1))
            continue
        m = re.search(r"side\s+(\d+)\s+sector\s+(\d+).*?first_phi\s+([-+0-9.eE]+).*?last_phi\s+([-+0-9.eE]+)", line)
        if m and region is not None and (region, int(m.group(1)), int(m.group(2))) not in data:
            side, sec = int(m.group(1)), int(m.group(2))
            first_phi, last_phi = float(m.group(3)), float(m.group(4))
            # Pad centers. Extend by half a pad pitch on both sides to get the active edge.
            delta = wrap_phi(last_phi - first_phi)
            npads = (94, 128, 192)[region]
            pitch = delta/(npads-1)
            edge0 = wrap_phi(first_phi - 0.5*pitch)
            edge1 = wrap_phi(last_phi + 0.5*pitch)
            data[(region, side, sec)] = (edge0, edge1)
    if len(data) != 3*2*12:
        raise RuntimeError(f"Expected 72 geometry entries, found {len(data)}")
    return data

def phi_in_interval(phi, edge0, edge1):
    width = (edge1-edge0) % (2*np.pi)
    return ((phi-edge0) % (2*np.pi)) <= width

def parse_gain_map(path, method=1):
    text = Path(path).read_text(errors="ignore")
    gain = np.ones((2, 12, 3), dtype=float)
    # Full-table columns end in Gain_m1 Gain_m2.
    pat = re.compile(r"^\s*([01])\s+(\d{1,2})\s+([123])\s+\d+\s+\d+\s+[-+0-9.eE]+\s+[yn]\s+[-+0-9.eE]+\s+[yn]\s+[-+0-9.eE]+\s+([-+0-9.eE]+)\s+([-+0-9.eE]+)\s*$")
    found = 0
    for line in text.splitlines():
        m = pat.match(line)
        if not m: continue
        side, sec, mod = int(m.group(1)), int(m.group(2)), int(m.group(3))-1
        gain[side, sec, mod] = float(m.group(4 if method == 1 else 5))
        found += 1
    if found != 72:
        raise RuntimeError(f"Expected 72 gain entries, found {found}")
    return gain

pad_geometry = parse_pad_geometry(PAD_PLACEMENT_FILE)
gain_map = parse_gain_map(GAIN_MAP_FILE, GAIN_METHOD)
print("Module radial bounds [cm]:\n", MODULE_R_BOUNDS_CM)
print("Gain range:", gain_map.min(), gain_map.max(), "mean:", gain_map.mean())


## Generate charge from spherical emission or flat pseudorapidity and optionally fit $C/r^\alpha$

In [ ]:
def eta_weight(eta, model):
    eta = np.asarray(eta, dtype=float)
    abs_eta = np.abs(eta)

    if model == "flat":
        weight = np.ones_like(eta)
    elif model == "double_hump":
        hump = np.exp(
            -0.5 * ((abs_eta - ETA_PEAK) / ETA_PEAK_WIDTH) ** 2
        )
        hump_shape = 1.0 + ETA_HUMP_AMPLITUDE * hump
        central_shape = (
            1.0
            - ETA_CENTRAL_DIP
            * np.exp(-0.5 * (eta / ETA_CENTRAL_WIDTH) ** 2)
        )
        edge_shape = 0.5 * (
            1.0
            - np.tanh((abs_eta - ETA_EDGE_START) / ETA_EDGE_WIDTH)
        )
        weight = hump_shape * central_shape * edge_shape
    else:
        raise ValueError(f"Unknown ETA_DISTRIBUTION={model!r}")

    maximum = np.max(weight)
    if maximum <= 0.0:
        raise RuntimeError("Eta model has no positive weight")
    return np.clip(weight / maximum, 0.0, 1.0)


def sample_eta_rejection(rng, n, eta_max, model):
    accepted = []
    n_accepted = 0

    while n_accepted < n:
        n_try = max(100_000, 2 * (n - n_accepted))
        eta_try = rng.uniform(-eta_max, eta_max, n_try)
        keep = rng.random(n_try) < eta_weight(eta_try, model)
        selected = eta_try[keep]
        accepted.append(selected)
        n_accepted += len(selected)

    return np.concatenate(accepted)[:n]


def generate_cylinder_radii(n, mode, z_plane_cm, eta_max, seed=12345):
    rng = np.random.default_rng(seed)

    if mode == "flat_eta":
        eta = rng.uniform(-eta_max, eta_max, n)
    elif mode == "realistic_eta":
        eta = sample_eta_rejection(
            rng, n, eta_max, ETA_DISTRIBUTION
        )
    elif mode == "isotropic_sphere":
        cos_theta = rng.uniform(-1.0, 1.0, n)
        theta = np.arccos(cos_theta)
        eta = -np.log(np.tan(theta / 2.0))
    else:
        raise ValueError(f"Unknown GENERATION_MODE={mode!r}")

    abs_eta = np.abs(eta)
    r = np.full_like(abs_eta, np.nan, dtype=float)
    valid_eta = abs_eta > 1.0e-12
    r[valid_eta] = z_plane_cm / np.sinh(abs_eta[valid_eta])
    phi = rng.uniform(-np.pi, np.pi, n)

    keep = (
        np.isfinite(r)
        & (r >= FIT_R_MIN_CM)
        & (r <= FIT_R_MAX_CM)
    )
    return r[keep], phi[keep], eta[keep], eta


def power_law(r, c, alpha):
    return c / np.power(r, alpha)


def fit_radial_distribution(radii, nbins=70):
    edges = np.linspace(FIT_R_MIN_CM, FIT_R_MAX_CM, nbins + 1)
    counts, _ = np.histogram(radii, bins=edges)
    centers = 0.5 * (edges[:-1] + edges[1:])
    widths = np.diff(edges)
    y = counts / widths
    sigma = np.sqrt(np.maximum(counts, 1.0)) / widths
    use = counts >= 20

    log_r = np.log(centers[use])
    log_y = np.log(y[use])
    slope, intercept = np.polyfit(log_r, log_y, 1)

    alpha0 = max(0.0, -slope)
    c0 = np.exp(intercept)

    popt, pcov = optimize.curve_fit(
        power_law,
        centers[use],
        y[use],
        p0=(c0, alpha0),
        sigma=sigma[use],
        absolute_sigma=True,
        bounds=([0.0, 0.0], [np.inf, 10.0]),
        maxfev=100_000,
    )
    return centers, y, sigma, use, popt, pcov


if GENERATE_RADIAL_STUDY:
    generated_r, generated_phi, generated_eta_active, generated_eta_all = (
        generate_cylinder_radii(
            N_GENERATED,
            GENERATION_MODE,
            Z_GEM_CM,
            ETA_MAX,
        )
    )

    centers, y, yerr, fit_mask, fitted, fitted_cov = (
        fit_radial_distribution(generated_r)
    )
    fitted_c, fitted_alpha = fitted
    fitted_alpha_error = np.sqrt(fitted_cov[1, 1])

    print(
        f"{GENERATION_MODE}: accepted {len(generated_r)} / {N_GENERATED}; "
        f"alpha = {fitted_alpha:.6f} +/- {fitted_alpha_error:.6f}"
    )

    eta_plot = np.linspace(-ETA_MAX, ETA_MAX, 1000)
    plt.figure(figsize=(8, 5))
    plt.hist(
        generated_eta_all,
        bins=120,
        range=(-ETA_MAX, ETA_MAX),
        histtype="step",
        label="Generated dN/deta",
    )
    model = eta_weight(eta_plot, ETA_DISTRIBUTION)
    hist_scale = N_GENERATED / 120.0
    plt.plot(eta_plot, model * hist_scale, label="Input eta shape")
    plt.xlabel(r"$\eta$")
    plt.ylabel("Entries")
    plt.legend()
    plt.tight_layout()
    plt.show()

    r_plot = np.linspace(
        centers[fit_mask].min(),
        centers[fit_mask].max(),
        500,
    )
    plt.figure(figsize=(8, 5))
    plt.errorbar(
        centers[fit_mask],
        y[fit_mask],
        yerr=yerr[fit_mask],
        fmt=".",
        label="Generated cylinder crossings",
    )
    plt.plot(
        r_plot,
        power_law(r_plot, fitted_c, fitted_alpha),
        label=rf"$C/r^{{\alpha}}$, $\alpha={fitted_alpha:.3f}$",
    )
    plt.yscale("log")
    plt.xlabel(r"$r$ [cm]")
    plt.ylabel(r"$dN/dr$")
    plt.title(f"Radial distribution: {GENERATION_MODE}")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    fitted_c, fitted_alpha = 1.0, FIXED_ALPHA[0]


if FIT_GENERATED_CHARGE:
    RADIAL_ALPHA = np.full(3, fitted_alpha, dtype=float)
    RADIAL_CONSTANT = np.full(3, fitted_c, dtype=float)
else:
    RADIAL_ALPHA = FIXED_ALPHA.copy()
    RADIAL_CONSTANT = FIXED_CONSTANT.copy()

print("RADIAL_ALPHA    =", RADIAL_ALPHA)
print("RADIAL_CONSTANT =", RADIAL_CONSTANT)


## Build signed-$z$ 3D source density using real GEM masks and gain multiplication

In [ ]:
a, b, L = A_CM*CM_TO_M, B_CM*CM_TO_M, L_CM*CM_TO_M
r_src_edges = np.linspace(a, b, NR_SRC+1)
phi_src_edges = np.linspace(-np.pi, np.pi, NPHI_SRC+1)
z_src_edges = np.linspace(-L, L, NZ_SRC+1)
r_src = 0.5*(r_src_edges[:-1]+r_src_edges[1:])
phi_src = 0.5*(phi_src_edges[:-1]+phi_src_edges[1:])
z_src = 0.5*(z_src_edges[:-1]+z_src_edges[1:])

dr = np.diff(r_src_edges)[:,None,None]
dphi = np.diff(phi_src_edges)[None,:,None]
dz = np.diff(z_src_edges)[None,None,:]
dV = r_src[:,None,None]*dr*dphi*dz

rho_shape = np.zeros((NR_SRC,NPHI_SRC,NZ_SRC), dtype=np.float64)
cell_gain = np.ones_like(rho_shape)
cell_module = -np.ones_like(rho_shape, dtype=np.int8)

for ir, r_m in enumerate(r_src):
    r_cm = r_m*M_TO_CM
    module = next((m for m,(lo,hi) in enumerate(MODULE_R_BOUNDS_CM) if lo <= r_cm <= hi), None)
    if module is None: continue
    radial = RADIAL_CONSTANT[module] / r_cm**RADIAL_ALPHA[module]
    for ip, phi in enumerate(phi_src):
        for iz, z_m in enumerate(z_src):
            side = 0 if z_m > 0 else 1
            sector = next((s for s in range(12) if phi_in_interval(phi, *pad_geometry[(module,side,s)])), None)
            if sector is None: continue
            g = gain_map[side,sector,module] if MULTIPLY_CHARGE_BY_GAIN else 1.0
            rho_shape[ir,ip,iz] = radial*g
            cell_gain[ir,ip,iz] = g
            cell_module[ir,ip,iz] = module

active = rho_shape > 0
if not np.any(active): raise RuntimeError("No active GEM/source cells")
if NORMALIZE_GAIN_WEIGHTED_TOTAL and MULTIPLY_CHARGE_BY_GAIN:
    # Remove only the overall gain-induced normalization while preserving the r model and asymmetry.
    base = np.zeros_like(rho_shape)
    base[active] = rho_shape[active]/cell_gain[active]
    norm = np.sum(base*dV)/np.sum(rho_shape*dV)
    rho_shape *= norm

target_rho = K_EFF*RHO_REFERENCE_NC_PER_M3*1e-9
rho = target_rho*rho_shape/(np.sum(rho_shape*dV)/np.sum(dV[active]))
charge = rho*dV
print("Active fraction:", active.mean())
print("Total charge [C]:", charge.sum())


## Modal Green-function preparation

In [ ]:
def F(k,m):
    return special.jv(m,k*b)*special.yv(m,k*a)-special.yv(m,k*b)*special.jv(m,k*a)
def Rfun(k,m,r):
    return special.jv(m,k*r)*special.yv(m,k*a)-special.yv(m,k*r)*special.jv(m,k*a)
def dRfun(k,m,r):
    return k*(special.jvp(m,k*r,1)*special.yv(m,k*a)-special.yvp(m,k*r,1)*special.jv(m,k*a))
def roots_for_m(m,nroot):
    roots=[]; step=0.2*np.pi/(b-a); x=max(1e-6,0.1*np.pi/(b-a)); fx=F(x,m)
    xmax=(nroot+m+30)*np.pi/(b-a)
    while x+step <= xmax and len(roots)<nroot:
        y=x+step; fy=F(y,m)
        if np.isfinite(fx) and np.isfinite(fy) and fx*fy<0:
            rt=optimize.brentq(F,x,y,args=(m,))
            if not roots or abs(rt-roots[-1])>1e-9: roots.append(rt)
        x,fx=y,fy
    if len(roots)!=nroot: raise RuntimeError(f"Only {len(roots)} roots for m={m}")
    return np.asarray(roots)

def radial_norm(k,m):
    val,_=integrate.quad(lambda rr: rr*Rfun(k,m,rr)**2, a, b, epsabs=1e-10, epsrel=1e-8, limit=200)
    return val

k_modes={}; radial_norms={}
for m in range(M_PHI_MAX+1):
    k_modes[m]=roots_for_m(m,N_RADIAL_MODES)
    radial_norms[m]=np.array([radial_norm(k,m) for k in k_modes[m]])
q_modes=np.arange(1,N_LONGITUDINAL_MODES+1)*np.pi/(2*L)


## Build sparse source arrays and precompute source-side basis tables

Only nonzero charge cells are stored. Source-side radial, azimuthal, and longitudinal basis values are precomputed once to avoid repeating expensive special-function evaluations in every Condor job.


In [ ]:
# Sparse source coordinates and charges.
Rsrc, Psrc, Zsrc = np.meshgrid(
    r_src,
    phi_src,
    z_src,
    indexing="ij",
)

src_r = Rsrc.ravel()
src_phi = Psrc.ravel()
src_z = Zsrc.ravel()
src_q = charge.ravel()

source_keep = src_q != 0.0
src_r = np.ascontiguousarray(src_r[source_keep], dtype=np.float64)
src_phi = np.ascontiguousarray(src_phi[source_keep], dtype=np.float64)
src_z = np.ascontiguousarray(src_z[source_keep], dtype=np.float64)
src_q = np.ascontiguousarray(src_q[source_keep], dtype=np.float64)

n_sources = len(src_q)
print(f"Nonzero source cells: {n_sources}")

# Observation axes; C++ reconstructs the flattened observation index using:
# index = (ir * NPHI_OBS + iphi) * NZ_OBS + iz
r_obs_edges = np.linspace(a, b, NR_OBS + 1)
phi_obs_edges = np.linspace(-np.pi, np.pi, NPHI_OBS + 1)
z_obs_edges = np.linspace(-L, L, NZ_OBS + 1)

r_obs = 0.5 * (r_obs_edges[:-1] + r_obs_edges[1:])
phi_obs = 0.5 * (phi_obs_edges[:-1] + phi_obs_edges[1:])
z_obs = 0.5 * (z_obs_edges[:-1] + z_obs_edges[1:])

n_observations = NR_OBS * NPHI_OBS * NZ_OBS
print(f"Observation points: {n_observations}")

# Flatten radial modes into one list and retain m and local radial-mode index.
mode_m = []
mode_ik = []
mode_k = []
mode_norm = []

for m in range(M_PHI_MAX + 1):
    for ik, (k, norm) in enumerate(zip(k_modes[m], radial_norms[m])):
        mode_m.append(m)
        mode_ik.append(ik)
        mode_k.append(k)
        mode_norm.append(norm)

mode_m = np.asarray(mode_m, dtype=np.int32)
mode_ik = np.asarray(mode_ik, dtype=np.int32)
mode_k = np.asarray(mode_k, dtype=np.float64)
mode_norm = np.asarray(mode_norm, dtype=np.float64)
q_modes_array = np.asarray(q_modes, dtype=np.float64)

n_mk = len(mode_k)
print(f"Flattened (m,k) modes: {n_mk}")
print(f"Longitudinal q modes: {len(q_modes_array)}")

source_basis = {}

if PRECOMPUTE_SOURCE_BASIS:
    print("Precomputing source radial basis...")
    radial_src = np.empty((n_mk, n_sources), dtype=SOURCE_BASIS_DTYPE)
    for imode, (m, k) in enumerate(zip(mode_m, mode_k)):
        radial_src[imode] = Rfun(k, int(m), src_r).astype(
            SOURCE_BASIS_DTYPE,
            copy=False,
        )

    print("Precomputing source phi basis...")
    m_values = np.arange(M_PHI_MAX + 1, dtype=np.float64)
    cos_mphi_src = np.cos(m_values[:, None] * src_phi[None, :]).astype(
        SOURCE_BASIS_DTYPE
    )
    sin_mphi_src = np.sin(m_values[:, None] * src_phi[None, :]).astype(
        SOURCE_BASIS_DTYPE
    )

    print("Precomputing source longitudinal basis...")
    us = src_z + L
    sin_qu_src = np.sin(q_modes_array[:, None] * us[None, :]).astype(
        SOURCE_BASIS_DTYPE
    )

    source_basis = {
        "radial_src": radial_src,
        "cos_mphi_src": cos_mphi_src,
        "sin_mphi_src": sin_mphi_src,
        "sin_qu_src": sin_qu_src,
    }

    total_basis_bytes = sum(arr.nbytes for arr in source_basis.values())
    print(
        f"Precomputed source basis size: "
        f"{total_basis_bytes / 1024**2:.1f} MiB"
    )


## Write the Condor input ROOT file

The file contains TTrees with scalar configuration, sparse source cells, mode tables, observation axes, and optional precomputed source-basis arrays. Large arrays are stored as one entry per row for simple C++ reading.


In [ ]:
from array import array


def write_vector_tree(directory, tree_name, branch_name, values, leaf_type):
    directory.cd()
    tree = ROOT.TTree(tree_name, tree_name)

    if leaf_type == "D":
        holder = array("d", [0.0])
    elif leaf_type == "F":
        holder = array("f", [0.0])
    elif leaf_type == "I":
        holder = array("i", [0])
    elif leaf_type == "L":
        holder = array("q", [0])
    else:
        raise ValueError(f"Unsupported leaf type {leaf_type!r}")

    tree.Branch(branch_name, holder, f"{branch_name}/{leaf_type}")
    for value in np.asarray(values).ravel():
        holder[0] = value
        tree.Fill()
    tree.Write()


def write_matrix_tree(directory, tree_name, matrix):
    """
    Store a 2D matrix as rows:
      row_index, values[n_columns]
    This is easy and efficient to read from C++.
    """
    matrix = np.asarray(matrix)
    if matrix.ndim != 2:
        raise ValueError("write_matrix_tree expects a 2D array")

    n_rows, n_columns = matrix.shape
    directory.cd()

    tree = ROOT.TTree(tree_name, tree_name)
    row_index = array("i", [0])
    values = array("f", [0.0] * n_columns)

    tree.Branch("row", row_index, "row/I")
    tree.Branch("values", values, f"values[{n_columns}]/F")

    for irow in range(n_rows):
        row_index[0] = irow
        row = matrix[irow]
        for icol in range(n_columns):
            values[icol] = float(row[icol])
        tree.Fill()

    tree.Write()


output_path = Path(CONDOR_INPUT_ROOT)
root_file = ROOT.TFile(str(output_path), "RECREATE")
if not root_file or root_file.IsZombie():
    raise RuntimeError(f"Could not create {output_path}")

# ----------------------------
# Scalar configuration
# ----------------------------
config_dir = root_file.mkdir("Config")
config_dir.cd()

config_tree = ROOT.TTree("config", "Condor field input configuration")

int_fields = {
    "format_version": 1,
    "nr_src": NR_SRC,
    "nphi_src": NPHI_SRC,
    "nz_src": NZ_SRC,
    "nr_obs": NR_OBS,
    "nphi_obs": NPHI_OBS,
    "nz_obs": NZ_OBS,
    "m_phi_max": M_PHI_MAX,
    "n_radial_modes": N_RADIAL_MODES,
    "n_longitudinal_modes": N_LONGITUDINAL_MODES,
    "n_sources": n_sources,
    "n_observations": n_observations,
    "n_mk_modes": n_mk,
    "gain_method": GAIN_METHOD,
    "multiply_charge_by_gain": int(MULTIPLY_CHARGE_BY_GAIN),
    "normalize_gain_weighted_total": int(NORMALIZE_GAIN_WEIGHTED_TOTAL),
    "precomputed_source_basis": int(PRECOMPUTE_SOURCE_BASIS),
}

double_fields = {
    "a_m": a,
    "b_m": b,
    "half_length_m": L,
    "epsilon0": EPSILON_0,
    "rho_reference_nc_per_m3": RHO_REFERENCE_NC_PER_M3,
    "k_eff": K_EFF,
    "total_charge_c": float(src_q.sum()),
}

int_holders = {name: array("i", [int(value)]) for name, value in int_fields.items()}
double_holders = {
    name: array("d", [float(value)])
    for name, value in double_fields.items()
}

for name, holder in int_holders.items():
    config_tree.Branch(name, holder, f"{name}/I")
for name, holder in double_holders.items():
    config_tree.Branch(name, holder, f"{name}/D")

config_tree.Fill()
config_tree.Write()

ROOT.TNamed("flattening_order", "(ir*NPHI_OBS + iphi)*NZ_OBS + iz").Write()
ROOT.TNamed("field_units", "V/m").Write()
ROOT.TNamed("coordinate_units", "m and radians").Write()
ROOT.TNamed("charge_units", "C").Write()

# ----------------------------
# Sparse source cells
# ----------------------------
sources_dir = root_file.mkdir("Sources")
sources_dir.cd()

source_tree = ROOT.TTree("sources", "Nonzero source charge cells")
r_h = array("d", [0.0])
phi_h = array("d", [0.0])
z_h = array("d", [0.0])
q_h = array("d", [0.0])

source_tree.Branch("r_m", r_h, "r_m/D")
source_tree.Branch("phi_rad", phi_h, "phi_rad/D")
source_tree.Branch("z_m", z_h, "z_m/D")
source_tree.Branch("charge_c", q_h, "charge_c/D")

for r_value, phi_value, z_value, q_value in zip(
    src_r, src_phi, src_z, src_q
):
    r_h[0] = float(r_value)
    phi_h[0] = float(phi_value)
    z_h[0] = float(z_value)
    q_h[0] = float(q_value)
    source_tree.Fill()

source_tree.Write()

# ----------------------------
# Observation axes
# ----------------------------
axes_dir = root_file.mkdir("ObservationAxes")
write_vector_tree(axes_dir, "r_centers", "value", r_obs, "D")
write_vector_tree(axes_dir, "phi_centers", "value", phi_obs, "D")
write_vector_tree(axes_dir, "z_centers", "value", z_obs, "D")
write_vector_tree(axes_dir, "r_edges", "value", r_obs_edges, "D")
write_vector_tree(axes_dir, "phi_edges", "value", phi_obs_edges, "D")
write_vector_tree(axes_dir, "z_edges", "value", z_obs_edges, "D")

# ----------------------------
# Mode tables
# ----------------------------
modes_dir = root_file.mkdir("Modes")
modes_dir.cd()

mk_tree = ROOT.TTree("radial_modes", "Flattened radial modes")
m_h = array("i", [0])
ik_h = array("i", [0])
k_h = array("d", [0.0])
norm_h = array("d", [0.0])

mk_tree.Branch("m", m_h, "m/I")
mk_tree.Branch("ik", ik_h, "ik/I")
mk_tree.Branch("k_per_m", k_h, "k_per_m/D")
mk_tree.Branch("norm_m2", norm_h, "norm_m2/D")

for m_value, ik_value, k_value, norm_value in zip(
    mode_m, mode_ik, mode_k, mode_norm
):
    m_h[0] = int(m_value)
    ik_h[0] = int(ik_value)
    k_h[0] = float(k_value)
    norm_h[0] = float(norm_value)
    mk_tree.Fill()

mk_tree.Write()
write_vector_tree(modes_dir, "q_modes", "q_per_m", q_modes_array, "D")

# ----------------------------
# Optional precomputed source basis
# ----------------------------
if PRECOMPUTE_SOURCE_BASIS:
    basis_dir = root_file.mkdir("SourceBasis")
    write_matrix_tree(basis_dir, "radial_src", radial_src)
    write_matrix_tree(basis_dir, "cos_mphi_src", cos_mphi_src)
    write_matrix_tree(basis_dir, "sin_mphi_src", sin_mphi_src)
    write_matrix_tree(basis_dir, "sin_qu_src", sin_qu_src)

# ----------------------------
# Optional charge QA map
# ----------------------------
if SAVE_CHARGE_TH3:
    qa_dir = root_file.mkdir("QA")
    qa_dir.cd()

    h_charge = ROOT.TH3F(
        "hCharge",
        "Source-cell charge;R [m];#phi [rad];z [m]",
        NR_SRC,
        np.asarray(r_src_edges, dtype="d"),
        NPHI_SRC,
        np.asarray(phi_src_edges, dtype="d"),
        NZ_SRC,
        np.asarray(z_src_edges, dtype="d"),
    )

    for ir in range(NR_SRC):
        for iphi in range(NPHI_SRC):
            for iz in range(NZ_SRC):
                h_charge.SetBinContent(
                    ir + 1,
                    iphi + 1,
                    iz + 1,
                    float(charge[ir, iphi, iz]),
                )

    h_charge.Write()

root_file.Write()
root_file.Close()

size_mib = output_path.stat().st_size / 1024**2
print(f"Wrote {output_path.resolve()}")
print(f"File size: {size_mib:.1f} MiB")


## Inspect the generated file


In [ ]:
f_check = ROOT.TFile.Open(CONDOR_INPUT_ROOT, "READ")
if not f_check or f_check.IsZombie():
    raise RuntimeError("Failed to reopen Condor input file")

f_check.ls()

config_check = f_check.Get("Config/config")
config_check.GetEntry(0)

print("n_sources      =", config_check.n_sources)
print("n_observations =", config_check.n_observations)
print("m_phi_max      =", config_check.m_phi_max)
print("radial modes   =", config_check.n_radial_modes)
print("longitudinal   =", config_check.n_longitudinal_modes)
print("total charge C =", config_check.total_charge_c)

f_check.Close()
